# MEDALLIO — CEO AI CONTROL TOWER LITE
## Fast Executive Control Tower · edición automatizable
### Notebook corporativo para Dirección, Data, Analytics, ML, Forecasting, Decision Intelligence y Gobierno

> **Objetivo:** convertir `medallio_dw` en un sistema corporativo de inteligencia que conecte datos, modelos, decisiones, control y creación de valor.

Este notebook eleva Medallio desde un laboratorio de Data Science hacia una **plataforma de gestión de IA empresarial**, con cinco preguntas rectoras:

1. **¿Qué está ocurriendo?** — observabilidad y KPIs.
2. **¿Por qué está ocurriendo?** — analytics, econometría y causalidad.
3. **¿Qué ocurrirá probablemente?** — ML y forecasting.
4. **¿Qué deberíamos hacer?** — Decision Intelligence y escenarios.
5. **¿Cómo sabemos que funcionó?** — experimentación, outcomes, ROI y gobierno.

---

## Arquitectura de valor

```text
FUENTES / CRM / MERCADO / FINANZAS / OPERACIÓN
                    │
                    ▼
          ┌──────────────────┐
          │ DATA FOUNDATION  │
          │ calidad · linaje │
          │ seguridad · SLA  │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │   ANALYTICS      │
          │ KPIs · cohortes  │
          │ mix · pricing    │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ ECONOMETRÍA / ML │
          │ explicación      │
          │ predicción       │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ DECISION INTEL.  │
          │ escenarios       │
          │ priorización     │
          └────────┬─────────┘
                   ▼
          ┌──────────────────┐
          │ VALUE REALIZATION│
          │ outcome · ROI    │
          │ aprendizaje      │
          └────────┬─────────┘
                   └──────────────► vuelve a aprender
```

---

## Estándar corporativo que persigue este notebook

- **Read-only por defecto.**
- **Evidence-first:** ninguna recomendación sin evidencia.
- **Time-aware:** splits y backtests temporales cuando corresponde.
- **Control de riesgo:** datos, modelos, seguridad y cambios.
- **Auditabilidad:** cada artefacto puede exportarse.
- **Valor medible:** toda iniciativa de IA debe vincularse con una decisión y un resultado económico/operativo.
- **Human-in-the-loop:** IA recomienda; la gobernanza define quién aprueba y quién responde.

---

### Navegación recomendada

1. Ejecuta **Configuración y conexión**.
2. Recorre primero **CEO AI CONTROL TOWER**.
3. Usa el resto del notebook para hacer drill-down técnico, econométrico y de ML.


# 0. Cómo ejecutar

Desde el repositorio:

```powershell
cd C:\Projectsd_replica_crm
.\.venv\Scripts\Activate.ps1
python -m pip install jupyterlab ipykernel pandas numpy matplotlib scikit-learn "psycopg[binary]" openpyxl pyarrow nbformat
python -m jupyter lab
```

Kernel recomendado: **Python - Medallio DW**.

In [ ]:
# Opcional: dependencias corporativas adicionales
# %pip install statsmodels scipy

# 1. Configuración corporativa y conexión segura

In [ ]:
from __future__ import annotations

import os, re, json, math, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RUN_TS = datetime.now(timezone.utc)
ARTIFACT_DIR = Path("artifacts") / "medallio_enterprise"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Run UTC:", RUN_TS.isoformat())
print("Artifacts:", ARTIFACT_DIR.resolve())

In [ ]:
def find_repo_root(start=None):
    p = (Path(start) if start else Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / ".git").exists() or (candidate / ".env.example").exists():
            return candidate
    return p

ROOT = find_repo_root()
ENV_PATH = ROOT / ".env"

def load_dotenv_minimal(path):
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        raw = raw.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        k, v = raw.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_dotenv_minimal(ENV_PATH)

DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "dbname": os.getenv("POSTGRES_DATABASE", "medallio_dw"),
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
    "connect_timeout": int(os.getenv("POSTGRES_CONNECT_TIMEOUT", "10")),
}

print("Repo:", ROOT)
print(f"DB: {DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
print("Password configurado:", bool(DB_CONFIG["password"]))

In [ ]:
import psycopg

FORBIDDEN_SQL = re.compile(
    r"\b(insert|update|delete|drop|alter|truncate|create|grant|revoke|copy|call|do)\b",
    flags=re.I,
)

def get_conn():
    return psycopg.connect(**DB_CONFIG)

def query_df(sql, params=None, read_only=True):
    statement = sql.strip().rstrip(";")
    if read_only and FORBIDDEN_SQL.search(statement):
        raise ValueError("Guardrail corporativo read-only: operación de escritura bloqueada.")
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(statement, params or ())
            cols = [d.name for d in cur.description] if cur.description else []
            rows = cur.fetchall() if cur.description else []
    return pd.DataFrame(rows, columns=cols)

def safe_relation(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Relación no permitida: {name}")
    return name

def relation_exists(name):
    schema, rel = safe_relation(name).split(".", 1)
    q = """
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s
        UNION ALL
        SELECT 1 FROM information_schema.views WHERE table_schema=%s AND table_name=%s
    ) AS ok
    """
    return bool(query_df(q, (schema, rel, schema, rel)).iloc[0, 0])

def columns_of(rel):
    schema, table = safe_relation(rel).split(".", 1)
    return query_df("""
        SELECT ordinal_position, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema=%s AND table_name=%s
        ORDER BY ordinal_position
    """, (schema, table))

try:
    display(query_df("SELECT current_database() AS database, now() AS postgres_now"))
except Exception as e:
    print("No se pudo conectar todavía:", type(e).__name__, e)

# CEO AI CONTROL TOWER
## Dashboard ejecutivo por cada capa de Medallio

Esta sección está diseñada para ser la **primera pantalla que vea Dirección** después de conectar el notebook.

No intenta reemplazar Power BI. Su función es otra:

- concentrar el estado de toda la plataforma de IA;
- mostrar dónde existe valor, riesgo o falta de evidencia;
- permitir drill-down hacia datasets, modelos y decisiones;
- servir como **war room / comité de IA**;
- convertir el notebook en una pieza presentable para Gerencia General.

### Capas visibles en la torre

1. **Data Foundation** — disponibilidad, calidad, volumen y cobertura.
2. **Business Analytics** — actividad comercial, proyectos y evolución.
3. **Econometría** — estacionalidad, comparabilidad y señales estructurales.
4. **Machine Learning** — readiness, targets, features y capacidad predictiva.
5. **Forecasting** — exactitud, sesgo, cobertura e incertidumbre.
6. **Decision Intelligence** — señales, prioridades, acciones y owners.
7. **MLOps & Governance** — drift, modelos, contratos y controles.
8. **Value Realization** — outcomes, ROI y madurez del portfolio.

> **Uso recomendado:** `Run All Above` hasta completar esta Control Tower.  
> Las secciones posteriores del notebook quedan como laboratorio y evidencia técnica.

# A. Motor visual de la Control Tower

In [ ]:
from IPython.display import HTML, display

def ctl_title(title, subtitle=None):
    sub = f"<div style='font-size:13px;opacity:.72;margin-top:4px'>{subtitle}</div>" if subtitle else ""
    display(HTML(
        f"<div style='margin:24px 0 12px 0'>"
        f"<div style='font-size:26px;font-weight:750;letter-spacing:-.4px'>{title}</div>{sub}</div>"
    ))

def ctl_kpis(items, columns=4):
    """
    items = [(label, value, detail), ...]
    """
    cards = []
    for label, value, detail in items:
        cards.append(
            "<div style='border:1px solid rgba(127,127,127,.28);"
            "border-radius:14px;padding:15px 16px;min-height:82px'>"
            f"<div style='font-size:12px;opacity:.68'>{label}</div>"
            f"<div style='font-size:27px;font-weight:760;margin:5px 0'>{value}</div>"
            f"<div style='font-size:11px;opacity:.65'>{detail}</div>"
            "</div>"
        )
    html = (
        f"<div style='display:grid;grid-template-columns:repeat({columns},minmax(0,1fr));"
        "gap:10px;margin:8px 0 18px 0'>" + "".join(cards) + "</div>"
    )
    display(HTML(html))

def ctl_callout(text, level="info"):
    display(HTML(
        "<div style='border-left:4px solid currentColor;padding:10px 14px;"
        "margin:10px 0;border-radius:6px;background:rgba(127,127,127,.08)'>"
        f"<b>{level.upper()}</b> · {text}</div>"
    ))

def ctl_table(df, caption=None, max_rows=20):
    if df is None or not len(df):
        ctl_callout("No hay datos disponibles para esta vista.", "N/D")
        return
    if caption:
        display(HTML(f"<div style='font-weight:650;margin:8px 0'>{caption}</div>"))
    display(df.head(max_rows).style.hide(axis="index"))

def ctl_finish(filename=None):
    plt.tight_layout()
    if filename:
        p = ARTIFACT_DIR / filename
        plt.savefig(p, dpi=160, bbox_inches="tight")
    plt.show()

print("✓ Motor visual CEO listo.")

# B. Snapshot corporativo de Medallio

### ⚡ Automation profile: FAST

Esta edición evita perfiles redundantes y consultas gigantes. Las vistas lentas se omiten
después de un `statement_timeout` local en lugar de bloquear todo el briefing.
El objetivo es que **Morning/Midday terminen en pocos minutos**, aun si una vista pesada
está temporalmente degradada.

In [ ]:

CTL_RELATIONS = [
    ("Data", "analytics.fact_evento_comercial"),
    ("Data", "analytics.snapshot_unidad_diario"),
    ("Data", "analytics.historial_oferta_unidad"),
    ("Data", "analytics.historial_etapa_proyecto"),
    ("Econometría", "analytics.v_econometria_cobertura"),
    ("Econometría", "analytics.v_econometria_serie_precios"),
    ("Econometría", "features.v_estacionalidad_panel"),
    ("ML", "features.v_dataset_unidad_entrenamiento"),
    ("ML", "features.v_contexto_mercado_por_corte"),
    ("Forecast", "features.commercial_forecast_snapshots"),
    ("Forecast", "model_control.commercial_forecast_runs"),
    ("Forecast", "analytics.commercial_forecast_predictions"),
    ("Forecast", "analytics.commercial_forecast_backtest"),
    ("Forecast", "analytics.v_commercial_forecast_current"),
    ("Forecast", "analytics.v_commercial_forecast_performance"),
    ("Governance", "analytics.v_commercial_forecast_coverage"),
    ("Governance", "analytics.v_commercial_forecast_monitoring"),
    ("Decision", "decision_intelligence.commercial_forecast_goals"),
    ("Decision", "decision_intelligence.commercial_forecast_actions"),
    ("Value", "analytics.commercial_forecast_outcomes"),
]

ctl_contract = pd.DataFrame(CTL_RELATIONS, columns=["layer", "object"])
ctl_contract["exists"] = ctl_contract["object"].map(
    lambda x: relation_exists(x) if "." in x else False
)
ctl_contract["status"] = np.where(ctl_contract["exists"], "ONLINE", "PENDING")

ctl_layer_coverage = (
    ctl_contract.groupby("layer", as_index=False)["exists"]
    .mean()
    .assign(coverage_pct=lambda d: (d["exists"] * 100).round(1))
)

# -------------------------------------------------------------------
# FAST MODE
# - No vuelve a consultar 20 objetos para perfilarlos.
# - Cada dataset se consulta una sola vez.
# - Cada consulta tiene statement_timeout.
# - Límites pequeños porque este notebook es una cabina ejecutiva,
#   no un proceso de entrenamiento.
# -------------------------------------------------------------------

def ctl_query_df(sql, params=None, statement_timeout_ms=30000):
    statement = sql.strip().rstrip(";")
    if FORBIDDEN_SQL.search(statement):
        raise ValueError("Guardrail read-only: escritura bloqueada.")
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SET LOCAL statement_timeout = {int(statement_timeout_ms)}")
            cur.execute(statement, params or ())
            cols = [d.name for d in cur.description] if cur.description else []
            rows = cur.fetchall() if cur.description else []
    return pd.DataFrame(rows, columns=cols)

try:
    ctl_catalog = ctl_query_df("""
        SELECT
            n.nspname AS schema_name,
            c.relname AS relation_name,
            CASE c.relkind
                WHEN 'r' THEN 'table'
                WHEN 'p' THEN 'partitioned_table'
                WHEN 'v' THEN 'view'
                WHEN 'm' THEN 'materialized_view'
                ELSE c.relkind::text
            END AS relation_type,
            pg_total_relation_size(c.oid) AS bytes_total,
            COALESCE(s.n_live_tup, 0) AS estimated_rows
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        LEFT JOIN pg_stat_user_tables s ON s.relid = c.oid
        WHERE n.nspname IN (
            'analytics','features','model_control','decision_intelligence',
            'staging','raw_cygnus','raw_mercado','etl_control','experiments'
        )
        AND c.relkind IN ('r','p','v','m')
    """, statement_timeout_ms=15000)
except Exception as e:
    print("Catalog omitido:", type(e).__name__)
    ctl_catalog = pd.DataFrame()

def ctl_fetch(rel, limit=10000, timeout_ms=30000):
    try:
        if relation_exists(rel):
            return ctl_query_df(
                f"SELECT * FROM {safe_relation(rel)} LIMIT {int(limit)}",
                statement_timeout_ms=timeout_ms,
            )
    except Exception as e:
        print(f"⏭ {rel}: omitido ({type(e).__name__})")
    return pd.DataFrame()

# Una sola lectura por dataset. Los límites son deliberadamente ejecutivos.
ctl_events = ctl_fetch("analytics.fact_evento_comercial", 50000, 45000)
ctl_season = ctl_fetch("features.v_estacionalidad_panel", 20000, 35000)
ctl_ml = ctl_fetch("features.v_dataset_unidad_entrenamiento", 20000, 35000)
ctl_forecast = ctl_fetch("analytics.v_commercial_forecast_current", 10000, 25000)
ctl_forecast_perf = ctl_fetch("analytics.v_commercial_forecast_performance", 15000, 25000)
ctl_forecast_monitor = ctl_fetch("analytics.v_commercial_forecast_monitoring", 10000, 20000)
ctl_actions = ctl_fetch("decision_intelligence.commercial_forecast_actions", 10000, 20000)
ctl_goals = ctl_fetch("decision_intelligence.commercial_forecast_goals", 10000, 20000)
ctl_outcomes = ctl_fetch("analytics.commercial_forecast_outcomes", 15000, 25000)
ctl_runs = ctl_fetch("model_control.commercial_forecast_runs", 10000, 20000)

# Perfil de calidad en memoria: cero round-trips adicionales.
_loaded = {
    "analytics.fact_evento_comercial": ctl_events,
    "features.v_estacionalidad_panel": ctl_season,
    "features.v_dataset_unidad_entrenamiento": ctl_ml,
    "analytics.v_commercial_forecast_current": ctl_forecast,
    "analytics.v_commercial_forecast_performance": ctl_forecast_perf,
    "analytics.v_commercial_forecast_monitoring": ctl_forecast_monitor,
    "decision_intelligence.commercial_forecast_actions": ctl_actions,
    "decision_intelligence.commercial_forecast_goals": ctl_goals,
    "analytics.commercial_forecast_outcomes": ctl_outcomes,
    "model_control.commercial_forecast_runs": ctl_runs,
}

def ctl_normalize_for_hash(v):
    """
    Convierte estructuras no-hashables (dict/list/set/ndarray) a texto estable
    para poder calcular duplicados sin romper el dashboard.
    """
    if isinstance(v, dict):
        try:
            return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)
        except Exception:
            return str(v)
    if isinstance(v, (list, tuple, set, np.ndarray)):
        try:
            return json.dumps(list(v), ensure_ascii=False, default=str)
        except Exception:
            return str(v)
    return v

def ctl_safe_duplicate_pct(df, sample_rows=5000):
    """
    Duplicate rate defensivo:
    - usa sólo una muestra acotada;
    - normaliza columnas object con valores anidados;
    - nunca bloquea la Control Tower por un tipo no hashable.
    """
    if df is None or not len(df):
        return np.nan

    work = df.head(min(len(df), int(sample_rows))).copy()

    for c in work.columns:
        if work[c].dtype == "object":
            # Sólo normalizar si detectamos estructuras anidadas.
            sample = work[c].dropna().head(50)
            if any(isinstance(v, (dict, list, tuple, set, np.ndarray)) for v in sample):
                work[c] = work[c].map(ctl_normalize_for_hash)

    try:
        return float(work.duplicated().mean() * 100)
    except TypeError:
        # Fallback ultraseguro: usa sólo columnas escalares hashables.
        safe_cols = []
        for c in work.columns:
            vals = work[c].dropna().head(50)
            ok = True
            for v in vals:
                try:
                    hash(v)
                except TypeError:
                    ok = False
                    break
            if ok:
                safe_cols.append(c)

        if not safe_cols:
            return np.nan

        return float(work[safe_cols].duplicated().mean() * 100)

ctl_profiles = []
for rel, df in _loaded.items():
    if df is None or not len(df):
        continue
    ctl_profiles.append({
        "object": rel,
        "rows_sample": len(df),
        "columns": df.shape[1],
        "null_pct": float(df.isna().mean().mean() * 100),
        "duplicate_pct": ctl_safe_duplicate_pct(df),
    })
ctl_profiles = pd.DataFrame(ctl_profiles)

def ctl_quality_status(row):
    n = row.get("null_pct", np.nan)
    d = row.get("duplicate_pct", np.nan)
    if pd.isna(n) and pd.isna(d):
        return "UNKNOWN"
    if (pd.notna(n) and n > 35) or (pd.notna(d) and d > 10):
        return "RED"
    if (pd.notna(n) and n > 15) or (pd.notna(d) and d > 3):
        return "AMBER"
    return "GREEN"

if len(ctl_profiles):
    ctl_profiles["quality_status"] = ctl_profiles.apply(ctl_quality_status, axis=1)

print(
    "✓ Snapshot corporativo FAST cargado:",
    RUN_TS.isoformat(),
    "| datasets cargados:",
    sum(bool(len(df)) for df in _loaded.values()),
    "/",
    len(_loaded),
)


# C. Utilidades de autodetección de columnas

In [ ]:
def ctl_find_col(df, candidates, contains=True):
    if df is None or not len(df):
        return None
    lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    if contains:
        for candidate in candidates:
            for c in df.columns:
                if candidate.lower() in c.lower():
                    return c
    return None

def ctl_numeric_candidates(df, hints=None):
    if df is None or not len(df):
        return []
    cols = list(df.select_dtypes(include=np.number).columns)
    if not hints:
        return cols
    preferred = [c for c in cols if any(h.lower() in c.lower() for h in hints)]
    return preferred + [c for c in cols if c not in preferred]

def ctl_date_series(df, col):
    if col is None:
        return pd.Series(pd.NaT, index=df.index)
    return pd.to_datetime(df[col], errors="coerce")

def ctl_pct(x):
    return "N/D" if pd.isna(x) else f"{x:.1f}%"

def ctl_num(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "N/D"
    if abs(float(x)) >= 1_000_000:
        return f"{float(x)/1_000_000:.1f}M"
    if abs(float(x)) >= 1_000:
        return f"{float(x)/1_000:.1f}K"
    return f"{float(x):,.0f}"

print("✓ Autodetección lista.")

# 1. CEO — AI CONTROL TOWER

## Una sola pantalla para responder:

- ¿Está sana la plataforma?
- ¿Dónde está el riesgo?
- ¿Dónde hay valor?
- ¿Los modelos están funcionando?
- ¿Qué decisión necesita atención?

In [ ]:
ctl_title(
    "Medallio AI Control Tower",
    "Vista CEO · salud de datos, IA, decisiones, riesgo y valor"
)

contract_pct = float(ctl_contract["exists"].mean() * 100) if len(ctl_contract) else np.nan
quality_green = (
    float((ctl_profiles["quality_status"] == "GREEN").mean() * 100)
    if len(ctl_profiles) and "quality_status" in ctl_profiles else np.nan
)

forecast_wape = np.nan
if len(ctl_forecast_perf):
    a_col = ctl_find_col(ctl_forecast_perf, ["actual", "y_actual", "real"])
    p_col = ctl_find_col(ctl_forecast_perf, ["prediction", "prediccion", "forecast", "y_pred"])
    if a_col and p_col:
        a = pd.to_numeric(ctl_forecast_perf[a_col], errors="coerce")
        p = pd.to_numeric(ctl_forecast_perf[p_col], errors="coerce")
        mask = a.notna() & p.notna()
        if mask.any():
            forecast_wape = float((a[mask] - p[mask]).abs().sum() / max(a[mask].abs().sum(), 1e-9) * 100)

drift_alerts = 0
if len(ctl_forecast_monitor):
    text = ctl_forecast_monitor.astype("string").apply(lambda s: s.str.lower())
    drift_alerts = int(text.apply(lambda s: s.str.contains("drift|alert|red|degrad", regex=True, na=False)).any(axis=1).sum())

pending_actions = len(ctl_actions)
if len(ctl_actions):
    status_col = ctl_find_col(ctl_actions, ["status", "estado"])
    if status_col:
        pending_actions = int(
            ~ctl_actions[status_col].astype("string").str.lower()
            .isin(["done", "closed", "completed", "completado", "cerrado"])
        .sum())

outcome_rows = len(ctl_outcomes)

ctl_kpis([
    ("Cobertura plataforma", ctl_pct(contract_pct), "objetos críticos disponibles"),
    ("Calidad observable", ctl_pct(quality_green), "datasets en estado GREEN"),
    ("Forecast WAPE", ctl_pct(forecast_wape), "error absoluto ponderado"),
    ("Alertas / drift", ctl_num(drift_alerts), "señales detectadas"),
    ("Acciones abiertas", ctl_num(pending_actions), "decision intelligence"),
    ("Outcomes capturados", ctl_num(outcome_rows), "evidencia para aprender"),
], columns=3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
d = ctl_layer_coverage.sort_values("coverage_pct")
ax.barh(d["layer"], d["coverage_pct"])
ax.set_xlim(0, 100)
ax.set_xlabel("% de capacidades técnicas disponibles")
ax.set_title("Readiness por capa de Medallio")
for i, v in enumerate(d["coverage_pct"]):
    ax.text(v + 1, i, f"{v:.0f}%", va="center")
ctl_finish("00_ceo_layer_readiness.png")

In [ ]:
if len(ctl_profiles) and "quality_status" in ctl_profiles:
    q = ctl_profiles["quality_status"].value_counts().reindex(["GREEN","AMBER","RED","UNKNOWN"], fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(q.index, q.values)
    ax.set_ylabel("Objetos")
    ax.set_title("Semáforo de calidad de datos")
    ctl_finish("00_ceo_data_quality.png")
else:
    ctl_callout("Todavía no hay perfiles de calidad suficientes.", "DATA")

In [ ]:
if len(ctl_catalog):
    s = (ctl_catalog.groupby("schema_name", as_index=False)
         .agg(bytes_total=("bytes_total","sum"), objects=("relation_name","count")))
    s["GB"] = s["bytes_total"] / 1024**3
    s = s.sort_values("GB", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(s["schema_name"], s["GB"])
    ax.set_ylabel("GB")
    ax.set_title("Data Estate — almacenamiento por dominio")
    ax.tick_params(axis="x", rotation=35)
    ctl_finish("00_ceo_data_estate.png")

# 2. CAPA DATA FOUNDATION
### ¿La materia prima de IA está disponible, completa y controlada?

In [ ]:
ctl_title("01 · Data Foundation", "Disponibilidad · calidad · volumen · contratos")

data_objects = ctl_contract.query("layer == 'Data'")
data_cov = data_objects["exists"].mean()*100 if len(data_objects) else np.nan
objects_total = len(ctl_catalog) if len(ctl_catalog) else 0
rows_est = ctl_catalog["estimated_rows"].sum() if len(ctl_catalog) and "estimated_rows" in ctl_catalog else np.nan
gb_total = ctl_catalog["bytes_total"].sum()/1024**3 if len(ctl_catalog) and "bytes_total" in ctl_catalog else np.nan

ctl_kpis([
    ("Cobertura Data", ctl_pct(data_cov), "objetos core online"),
    ("Objetos DW", ctl_num(objects_total), "tablas + vistas observables"),
    ("Filas estimadas", ctl_num(rows_est), "pg_stat_user_tables"),
    ("Huella DW", f"{gb_total:.2f} GB" if pd.notna(gb_total) else "N/D", "almacenamiento observado"),
], columns=4)

In [ ]:
if len(ctl_profiles):
    top_nulls = ctl_profiles.sort_values("null_pct", ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top_nulls["object"], top_nulls["null_pct"])
    ax.invert_yaxis()
    ax.set_xlabel("% nulos promedio en muestra")
    ax.set_title("Data Quality — objetos con mayor presión de nulos")
    ctl_finish("01_data_null_pressure.png")

In [ ]:
if len(ctl_catalog):
    obj = (ctl_catalog.groupby("schema_name", as_index=False)["relation_name"]
           .count().rename(columns={"relation_name":"objects"}).sort_values("objects"))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(obj["schema_name"], obj["objects"])
    ax.set_xlabel("Objetos")
    ax.set_title("Cobertura física por esquema")
    ctl_finish("01_data_objects_schema.png")

# 3. CAPA BUSINESS ANALYTICS
### ¿Qué está cambiando comercialmente?

In [ ]:
ctl_title("02 · Business Analytics", "Actividad · proyectos · evolución temporal")

if len(ctl_events):
    date_col = ctl_find_col(ctl_events, ["fecha_evento","fecha","fecha_inicio","created_at","captured_at"])
    project_col = ctl_find_col(ctl_events, ["nombre_proyecto","codigo_proyecto","proyecto"])
    event_col = ctl_find_col(ctl_events, ["tipo_evento","evento","nombre_evento","nombre_proceso","nombre"])

    ctl_kpis([
        ("Eventos observados", ctl_num(len(ctl_events)), "muestra / vista comercial"),
        ("Proyectos", ctl_num(ctl_events[project_col].nunique()) if project_col else "N/D", "entidades comerciales"),
        ("Tipos de evento", ctl_num(ctl_events[event_col].nunique()) if event_col else "N/D", "señales de funnel"),
        ("Desde", str(ctl_date_series(ctl_events,date_col).min().date()) if date_col and ctl_date_series(ctl_events,date_col).notna().any() else "N/D", "inicio observado"),
    ], columns=4)
else:
    ctl_callout("fact_evento_comercial todavía no está disponible.", "ANALYTICS")

In [ ]:
if len(ctl_events):
    date_col = ctl_find_col(ctl_events, ["fecha_evento","fecha","fecha_inicio","created_at","captured_at"])
    if date_col:
        tmp = ctl_events.copy()
        tmp["_date"] = ctl_date_series(tmp, date_col)
        tmp = tmp.dropna(subset=["_date"])
        if len(tmp):
            monthly = tmp.groupby(tmp["_date"].dt.to_period("M")).size()
            x = monthly.index.astype(str)
            fig, ax = plt.subplots(figsize=(11, 4))
            ax.plot(x, monthly.values, marker="o")
            ax.set_title("Actividad comercial observada por mes")
            ax.set_ylabel("Eventos")
            ax.tick_params(axis="x", rotation=60)
            ctl_finish("02_analytics_monthly_events.png")

In [ ]:
if len(ctl_events):
    project_col = ctl_find_col(ctl_events, ["nombre_proyecto","codigo_proyecto","proyecto"])
    if project_col:
        p = ctl_events[project_col].astype("string").value_counts().head(15).sort_values()
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(p.index, p.values)
        ax.set_xlabel("Eventos")
        ax.set_title("Top proyectos por actividad observable")
        ctl_finish("02_analytics_projects.png")

In [ ]:
if len(ctl_events):
    event_col = ctl_find_col(ctl_events, ["tipo_evento","evento","nombre_evento","nombre_proceso","nombre"])
    if event_col:
        e = ctl_events[event_col].astype("string").value_counts().head(12).sort_values()
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(e.index, e.values)
        ax.set_xlabel("Eventos")
        ax.set_title("Mix de eventos comerciales")
        ctl_finish("02_analytics_event_mix.png")

# 4. CAPA ECONOMETRÍA
### ¿Qué parte del resultado es proyecto, ciclo de vida, mix o calendario?

In [ ]:
ctl_title("03 · Econometría", "Comparabilidad interproyecto · estacionalidad · elasticidad")

if len(ctl_season):
    nums = ctl_numeric_candidates(
        ctl_season,
        ["venta","minuta","separ","absor","ritmo","stock","demanda","precio"]
    )
    ctl_kpis([
        ("Filas panel", ctl_num(len(ctl_season)), "observaciones econométricas"),
        ("Variables", ctl_num(ctl_season.shape[1]), "campos disponibles"),
        ("Métricas numéricas", ctl_num(len(nums)), "candidatas"),
        ("Cobertura nulos", ctl_pct(100 - ctl_season.isna().mean().mean()*100), "completitud media"),
    ], columns=4)
else:
    ctl_callout("v_estacionalidad_panel aún no está disponible.", "ECONOMETRÍA")

In [ ]:
if len(ctl_season):
    nums = ctl_numeric_candidates(ctl_season, ["venta","minuta","separ","absor","ritmo","stock","demanda"])
    if nums:
        metric = nums[0]
        month_col = ctl_find_col(ctl_season, ["mes_calendario","mes","month","periodo_mes"])
        if month_col:
            t = ctl_season[[month_col, metric]].copy()
            t[metric] = pd.to_numeric(t[metric], errors="coerce")
            agg = t.groupby(month_col, dropna=False)[metric].mean().dropna()
            if len(agg):
                fig, ax = plt.subplots(figsize=(9, 4))
                ax.bar(agg.index.astype(str), agg.values)
                ax.set_title(f"Perfil estacional — {metric}")
                ax.set_ylabel("Promedio")
                ax.tick_params(axis="x", rotation=45)
                ctl_finish("03_econometrics_seasonality.png")

In [ ]:
if len(ctl_season):
    nums = ctl_season.select_dtypes(include=np.number)
    if nums.shape[1] >= 2:
        cols = nums.var().sort_values(ascending=False).head(12).index
        corr = nums[cols].corr()
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(corr.to_numpy(), aspect="auto", vmin=-1, vmax=1)
        ax.set_xticks(range(len(cols)), cols, rotation=90)
        ax.set_yticks(range(len(cols)), cols)
        ax.set_title("Mapa de relaciones — panel econométrico")
        fig.colorbar(im, ax=ax)
        ctl_finish("03_econometrics_correlations.png")

# 5. CAPA MACHINE LEARNING
### ¿Tenemos un dataset realmente entrenable y señales con potencial predictivo?

In [ ]:
ctl_title("04 · Machine Learning", "Readiness · features · target · señal predictiva")

if len(ctl_ml):
    target_hints = [
        c for c in ctl_ml.columns
        if any(h in c.lower() for h in ["target","venta","vendido","minuta","separacion","conversion"])
    ]
    numeric = ctl_ml.select_dtypes(include=np.number).columns
    ctl_kpis([
        ("Filas entrenables", ctl_num(len(ctl_ml)), "muestra del dataset"),
        ("Features", ctl_num(ctl_ml.shape[1]), "columnas disponibles"),
        ("Numéricas", ctl_num(len(numeric)), "features cuantitativas"),
        ("Targets candidatos", ctl_num(len(target_hints)), "requieren definición"),
    ], columns=4)
else:
    ctl_callout("v_dataset_unidad_entrenamiento aún no está disponible.", "ML")

In [ ]:
if len(ctl_ml):
    miss = (ctl_ml.isna().mean()*100).sort_values(ascending=False).head(20).sort_values()
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(miss.index, miss.values)
    ax.set_xlabel("% nulos")
    ax.set_title("ML Readiness — missingness de features")
    ctl_finish("04_ml_missingness.png")

In [ ]:
if len(ctl_ml):
    nums = ctl_ml.select_dtypes(include=np.number)
    candidates = [c for c in nums.columns if nums[c].nunique(dropna=True) > 5]
    if candidates:
        v = candidates[0]
        s = pd.to_numeric(ctl_ml[v], errors="coerce").dropna()
        if len(s):
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(s, bins=35)
            ax.set_title(f"Distribución de feature — {v}")
            ax.set_xlabel(v)
            ax.set_ylabel("Frecuencia")
            ctl_finish("04_ml_feature_distribution.png")

# 6. CAPA FORECASTING
### ¿Qué creemos que ocurrirá y qué tan confiable ha sido esa creencia?

In [ ]:
ctl_title("05 · Forecasting", "Exactitud · sesgo · horizonte · cobertura")

wape = bias = mae = np.nan
mature = 0

if len(ctl_forecast_perf):
    a_col = ctl_find_col(ctl_forecast_perf, ["actual","real","y_actual"])
    p_col = ctl_find_col(ctl_forecast_perf, ["prediction","prediccion","forecast","y_pred"])
    if a_col and p_col:
        a = pd.to_numeric(ctl_forecast_perf[a_col], errors="coerce")
        p = pd.to_numeric(ctl_forecast_perf[p_col], errors="coerce")
        mask = a.notna() & p.notna()
        mature = int(mask.sum())
        if mature:
            err = p[mask] - a[mask]
            mae = float(err.abs().mean())
            bias = float(err.mean())
            wape = float(err.abs().sum()/max(a[mask].abs().sum(),1e-9)*100)

ctl_kpis([
    ("Forecasts vigentes", ctl_num(len(ctl_forecast)), "current forecast"),
    ("Outcomes maduros", ctl_num(mature), "actual vs prediction"),
    ("WAPE", ctl_pct(wape), "menor es mejor"),
    ("MAE", f"{mae:.2f}" if pd.notna(mae) else "N/D", "error medio absoluto"),
    ("Bias", f"{bias:+.2f}" if pd.notna(bias) else "N/D", "sobre/sub predicción"),
], columns=5)

In [ ]:
if len(ctl_forecast_perf):
    a_col = ctl_find_col(ctl_forecast_perf, ["actual","real","y_actual"])
    p_col = ctl_find_col(ctl_forecast_perf, ["prediction","prediccion","forecast","y_pred"])
    if a_col and p_col:
        a = pd.to_numeric(ctl_forecast_perf[a_col], errors="coerce")
        p = pd.to_numeric(ctl_forecast_perf[p_col], errors="coerce")
        m = a.notna() & p.notna()
        if m.any():
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(a[m], p[m], alpha=.5)
            lo = min(a[m].min(), p[m].min())
            hi = max(a[m].max(), p[m].max())
            ax.plot([lo, hi], [lo, hi], linestyle="--")
            ax.set_xlabel("Real")
            ax.set_ylabel("Forecast")
            ax.set_title("Calibración visual — Real vs Forecast")
            ctl_finish("05_forecast_actual_vs_pred.png")

In [ ]:
if len(ctl_forecast_monitor):
    h_col = ctl_find_col(ctl_forecast_monitor, ["horizon","horizonte"])
    metric_candidates = [
        c for c in ctl_forecast_monitor.columns
        if any(k in c.lower() for k in ["wape","mae","bias"])
        and pd.api.types.is_numeric_dtype(ctl_forecast_monitor[c])
    ]
    if h_col and metric_candidates:
        metric = metric_candidates[0]
        temp = ctl_forecast_monitor[[h_col,metric]].dropna()
        if len(temp):
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(temp[h_col].astype(str), temp[metric], marker="o")
            ax.set_xlabel("Horizonte")
            ax.set_ylabel(metric)
            ax.set_title(f"Degradación por horizonte — {metric}")
            ctl_finish("05_forecast_horizon.png")

# 7. CAPA DECISION INTELLIGENCE
### ¿Qué debe hacerse, por quién y en qué orden?

In [ ]:
ctl_title("06 · Decision Intelligence", "Prioridad · acciones · metas · accountability")

status_col = ctl_find_col(ctl_actions, ["status","estado"]) if len(ctl_actions) else None
owner_col = ctl_find_col(ctl_actions, ["owner","responsable","usuario"]) if len(ctl_actions) else None
priority_col = ctl_find_col(ctl_actions, ["priority","prioridad","priority_score","impacto"]) if len(ctl_actions) else None

open_count = len(ctl_actions)
closed_count = 0
if status_col:
    st = ctl_actions[status_col].astype("string").str.lower()
    closed = st.isin(["done","closed","completed","completado","cerrado"])
    closed_count = int(closed.sum())
    open_count = int((~closed).sum())

ctl_kpis([
    ("Metas registradas", ctl_num(len(ctl_goals)), "decision goals"),
    ("Acciones abiertas", ctl_num(open_count), "requieren atención"),
    ("Acciones cerradas", ctl_num(closed_count), "ciclo completado"),
    ("Owners", ctl_num(ctl_actions[owner_col].nunique()) if owner_col else "N/D", "accountability"),
], columns=4)

In [ ]:
if len(ctl_actions) and status_col:
    vc = ctl_actions[status_col].astype("string").fillna("<NA>").value_counts().head(12).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(vc.index, vc.values)
    ax.set_xlabel("Acciones")
    ax.set_title("Estado de acciones recomendadas")
    ctl_finish("06_decision_action_status.png")
else:
    ctl_callout("Aún no hay suficiente historial de acciones para graficar estados.", "DECISION")

In [ ]:
if len(ctl_actions) and owner_col:
    vc = ctl_actions[owner_col].astype("string").fillna("<NA>").value_counts().head(12).sort_values()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(vc.index, vc.values)
    ax.set_xlabel("Acciones")
    ax.set_title("Carga de decisiones por owner")
    ctl_finish("06_decision_owner_load.png")

# 8. CAPA MLOPS & GOVERNANCE
### ¿Podemos confiar, auditar y revertir lo que hace la IA?

In [ ]:
ctl_title("07 · MLOps & Governance", "Runs · contratos · drift · control")

gov_cov = ctl_contract.query("layer == 'Governance'")["exists"].mean()*100 if len(ctl_contract.query("layer == 'Governance'")) else np.nan
runs_count = len(ctl_runs)

run_status_col = ctl_find_col(ctl_runs, ["status","estado","stage"]) if len(ctl_runs) else None
failed_runs = 0
if run_status_col:
    failed_runs = int(
        ctl_runs[run_status_col].astype("string").str.lower()
        .str.contains("fail|error|failed", regex=True, na=False).sum()
    )

ctl_kpis([
    ("Cobertura gobierno", ctl_pct(gov_cov), "objetos de monitoring"),
    ("Model runs", ctl_num(runs_count), "ejecuciones registradas"),
    ("Runs fallidos", ctl_num(failed_runs), "operación"),
    ("Drift / alertas", ctl_num(drift_alerts), "señales a revisar"),
], columns=4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
d = ctl_layer_coverage.sort_values("coverage_pct")
ax.barh(d["layer"], d["coverage_pct"])
ax.set_xlim(0,100)
ax.set_xlabel("%")
ax.set_title("Control de arquitectura — cobertura de contratos por capa")
ctl_finish("07_governance_contract_coverage.png")

In [ ]:
if len(ctl_runs) and run_status_col:
    vc = ctl_runs[run_status_col].astype("string").fillna("<NA>").value_counts().head(12).sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(vc.index, vc.values)
    ax.set_xlabel("Runs")
    ax.set_title("Model / pipeline runs por estado")
    ctl_finish("07_governance_run_status.png")

# 9. CAPA VALUE REALIZATION
### ¿La IA está generando valor comprobable?

In [ ]:
ctl_title("08 · Value Realization", "Outcome · adopción · ROI · aprendizaje")

value_layer_cov = ctl_contract.query("layer == 'Value'")["exists"].mean()*100 if len(ctl_contract.query("layer == 'Value'")) else np.nan

numeric_outcomes = ctl_outcomes.select_dtypes(include=np.number) if len(ctl_outcomes) else pd.DataFrame()
money_cols = [
    c for c in numeric_outcomes.columns
    if any(k in c.lower() for k in ["impact","valor","revenue","monto","benefit","beneficio","roi","uplift"])
]
observed_value = np.nan
if money_cols:
    observed_value = pd.to_numeric(ctl_outcomes[money_cols[0]], errors="coerce").sum()

ctl_kpis([
    ("Outcome coverage", ctl_pct(value_layer_cov), "infraestructura de resultados"),
    ("Outcomes", ctl_num(len(ctl_outcomes)), "decisiones con resultado"),
    ("Variables de valor", ctl_num(len(money_cols)), "campos monetarios/uplift"),
    ("Valor observable", ctl_num(observed_value) if pd.notna(observed_value) else "N/D", "según primer campo de valor"),
], columns=4)

In [ ]:
portfolio = pd.DataFrame([
    ["Forecast comercial", 9, 8, 7],
    ["Lead scoring", 8, 7, 6],
    ["Stock risk", 8, 8, 5],
    ["Pricing elasticity", 9, 5, 8],
    ["Anomaly detection", 6, 9, 3],
    ["Next best action", 9, 4, 8],
], columns=["use_case","business_value","data_readiness","risk_complexity"])
portfolio["priority"] = (
    portfolio["business_value"] * portfolio["data_readiness"]
    / (portfolio["risk_complexity"] + 1)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    portfolio["data_readiness"],
    portfolio["business_value"],
    s=(portfolio["risk_complexity"]+1)*55
)
for _, r in portfolio.iterrows():
    ax.text(r["data_readiness"]+.05, r["business_value"]+.03, r["use_case"], fontsize=8)
ax.set_xlim(0,10)
ax.set_ylim(0,10)
ax.set_xlabel("Data readiness")
ax.set_ylabel("Business value")
ax.set_title("AI Portfolio — valor vs readiness (tamaño = riesgo/complejidad)")
ctl_finish("08_value_ai_portfolio.png")

# 10. CEO HEATMAP — DÓNDE ESTAMOS HOY

La torre debe terminar en una lectura de **fortaleza / debilidad por capa**, no en cien métricas desconectadas.

In [ ]:
ctl_title("CEO Layer Scorecard", "Mapa de madurez observable de la fábrica de inteligencia")

layer_scores = ctl_layer_coverage[["layer","coverage_pct"]].copy()

if len(ctl_profiles):
    quality_score = (ctl_profiles["quality_status"] == "GREEN").mean()*100
else:
    quality_score = np.nan

layer_scores = pd.concat([
    layer_scores,
    pd.DataFrame([["Data Quality", quality_score]], columns=["layer","coverage_pct"])
], ignore_index=True)

display(
    layer_scores
    .sort_values("coverage_pct", ascending=False)
    .style
    .format({"coverage_pct":"{:.1f}%"})
    .bar(subset=["coverage_pct"])
    .hide(axis="index")
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
d = layer_scores.sort_values("coverage_pct")
ax.barh(d["layer"], d["coverage_pct"])
ax.set_xlim(0,100)
ax.set_xlabel("Score observable (%)")
ax.set_title("CEO Heatmap — madurez por capa")
for i, v in enumerate(d["coverage_pct"]):
    if pd.notna(v):
        ax.text(v + 1, i, f"{v:.0f}%", va="center")
ctl_finish("09_ceo_heatmap.png")

# 11. CEO ACTION BOARD

El dashboard termina en decisiones, no en gráficos.

La lectura correcta es:

> **Señal → Impacto → Confianza → Owner → Acción → Outcome**

In [ ]:
action_board = []

for _, r in ctl_layer_coverage.iterrows():
    if r["coverage_pct"] < 100:
        action_board.append({
            "priority": 100 - r["coverage_pct"],
            "layer": r["layer"],
            "signal": f"Cobertura incompleta: {r['coverage_pct']:.0f}%",
            "recommended_action": "Completar objetos/contratos críticos de la capa",
            "owner": "Data & AI"
        })

if pd.notna(forecast_wape) and forecast_wape > 25:
    action_board.append({
        "priority": min(forecast_wape,100),
        "layer": "Forecast",
        "signal": f"WAPE elevado: {forecast_wape:.1f}%",
        "recommended_action": "Revisar por horizonte/proyecto y challenger",
        "owner": "Data Science"
    })

if drift_alerts > 0:
    action_board.append({
        "priority": 85,
        "layer": "Governance",
        "signal": f"{drift_alerts} alertas/drift",
        "recommended_action": "Validar degradación antes de promover/reentrenar",
        "owner": "AI Governance"
    })

if pending_actions > 0:
    action_board.append({
        "priority": 70,
        "layer": "Decision",
        "signal": f"{pending_actions} acciones abiertas",
        "recommended_action": "Revisar owners, vencimientos y outcomes",
        "owner": "Product Owner"
    })

ctl_action_board = pd.DataFrame(action_board)
if len(ctl_action_board):
    ctl_action_board = ctl_action_board.sort_values("priority", ascending=False)
    ctl_table(ctl_action_board, "Prioridades sugeridas por la Control Tower", 20)
else:
    ctl_callout("No se detectaron brechas automáticas con las reglas actuales.", "OK")

In [ ]:
if len(ctl_action_board):
    plot = ctl_action_board.head(12).sort_values("priority")
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = plot["layer"] + " · " + plot["signal"]
    ax.barh(labels, plot["priority"])
    ax.set_xlabel("Priority score")
    ax.set_title("CEO Action Board — qué atender primero")
    ctl_finish("10_ceo_action_board.png")

# 12. Export CEO Pack

Los gráficos anteriores se guardan automáticamente en:

```text
artifacts/medallio_enterprise/
```

Esto permite:

- capturarlos directamente para comité;
- incorporarlos a un informe ejecutivo;
- comparar snapshots de la Control Tower;
- llevar sólo los componentes que maduren a Power BI.

### División recomendada Notebook vs Power BI

**Mantener en Notebook**
- diagnóstico de calidad;
- experimentos;
- econometría;
- feature importance;
- comparación challenger;
- drift;
- model cards;
- análisis de anomalías;
- escenarios ad hoc;
- CEO AI Control Tower para comité técnico/ejecutivo.

**Llevar a Power BI cuando madure**
- KPIs recurrentes;
- funnel comercial;
- stock/absorción;
- forecast operativo por proyecto;
- alertas para usuarios;
- seguimiento de metas;
- acciones por asesor/proyecto;
- indicadores con refresh programado.

Así el notebook queda como **cabina de inteligencia y laboratorio ejecutivo**, mientras Power BI funciona como la **capa operacional masiva**.

In [ ]:
ctl_summary = pd.DataFrame([{
    "run_utc": RUN_TS.isoformat(),
    "platform_coverage_pct": contract_pct,
    "quality_green_pct": quality_green,
    "forecast_wape_pct": forecast_wape,
    "drift_alerts": drift_alerts,
    "open_actions": pending_actions,
    "outcomes": outcome_rows,
}])

ctl_summary.to_csv(ARTIFACT_DIR / "ceo_control_tower_summary.csv", index=False)
ctl_layer_coverage.to_csv(ARTIFACT_DIR / "ceo_layer_coverage.csv", index=False)
ctl_contract.to_csv(ARTIFACT_DIR / "ceo_contract_status.csv", index=False)
if len(ctl_profiles):
    ctl_profiles.to_csv(ARTIFACT_DIR / "ceo_data_quality.csv", index=False)
if len(ctl_action_board):
    ctl_action_board.to_csv(ARTIFACT_DIR / "ceo_action_board.csv", index=False)

print("✓ CEO Pack exportado en:", ARTIFACT_DIR.resolve())